# Approfondimento 3 · Poche donne nei dati

**CAF GIAS, Università degli Studi Link, modulo 2.4** · Michele Ferramola · slide «Il modello minimizza l'errore medio» e «Prima prova: poche donne nei dati»

**Il messaggio.** Con meno donne nei dati l'AUROC resta fermo, ma il rischio delle donne viene sottostimato.
Il divario c'è già a metà e metà: lo squilibrio lo peggiora, non lo crea.

**I dati.** NHANES 2013-2014, 5.507 adulti dai 20 anni. L'artrite è più frequente nelle donne.
Versione semplificata dell'esperimento della slide: un solo modello per scenario, regressione logistica al posto di XGBoost.

### Passo 1 · Quanto è frequente l'artrite

**Atteso:** 31% nelle donne, 21% negli uomini.

In [ ]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

DATI = "https://raw.githubusercontent.com/MicheleFerramola00/lezioni/caf-unilink-m2.4/dati/"
D = pd.read_parquet(DATI + "artrite_nhanes_adulti.parquet")
D.groupby(D.donna.map({1: "donne", 0: "uomini"})).artrite.mean().round(2)

### Passo 2 · Quattro modelli, con il 50, 30, 15 e 5% di donne

Ogni modello impara su 1.500 pazienti e stima il rischio di artrite da età e BMI, senza usare il sesso.
Tutti vengono verificati sugli stessi 1.500 pazienti. Per ogni gruppo confrontiamo il rischio medio
che il modello dichiara con la frequenza vera dell'artrite.

**Atteso:** AUROC fermo, 0,83 nelle donne e 0,76 negli uomini. Per le donne di verifica il modello dichiara
il 27% con metà donne e il 23% con il 5%, contro il 33% vero. Per gli uomini dichiara dal 25% al 21%, contro il 21% vero.

In [ ]:
ver = D.sample(1500, random_state=0)
resto = D.drop(ver.index)
donne, uomini = resto[resto.donna == 1], resto[resto.donna == 0]
f = ver.donna == 1

righe = []
for quota in (0.50, 0.30, 0.15, 0.05):
    n_donne = int(1500 * quota)
    imp = pd.concat([donne.sample(n_donne, random_state=0), uomini.sample(1500 - n_donne, random_state=0)])
    rischio = LogisticRegression().fit(imp[["eta", "bmi"]], imp.artrite).predict_proba(ver[["eta", "bmi"]])[:, 1]
    righe.append((f"{quota:.0%}", roc_auc_score(ver.artrite[f], rischio[f]), roc_auc_score(ver.artrite[~f], rischio[~f]),
                  f"{rischio[f].mean():.0%} contro {ver.artrite[f].mean():.0%}",
                  f"{rischio[~f].mean():.0%} contro {ver.artrite[~f].mean():.0%}"))
pd.DataFrame(righe, columns=["donne nei dati", "AUROC donne", "AUROC uomini",
                             "donne: rischio dichiarato contro vero", "uomini: rischio dichiarato contro vero"]).round(2)

---

### Cosa abbiamo visto

1. **L'AUROC non si muove:** l'ordine dei pazienti resta buono in entrambi i gruppi.
2. **Il rischio delle donne viene sottostimato, e sempre di più:** il modello impara la regola del gruppo più numeroso.
3. **Il divario c'è già a metà e metà:** bilanciare i dati non basta. Un report con il solo AUROC non lo mostra.

### Da chiedere in gara

> **Risultati separati per gruppo, con quanti pazienti per gruppo: AUROC e calibrazione.**